# 05 — Kiểm tra chất lượng các bảng đã xử lý

## Mục tiêu kiểm tra trước khi phân tích

Notebook đọc tám bảng trong `data/processed/tables/` và kiểm tra các điều kiện dữ liệu trước khi thực hiện EDA. Mỗi bảng được khai báo rõ tên tệp, khóa, đơn vị quan sát và mục đích sử dụng.

Các kiểm tra ban đầu xác nhận khóa không thiếu, không trùng và các cột số không chứa giá trị vô cực. Nếu một điều kiện kiểm tra thất bại, notebook dừng tại bước đó.

Báo cáo chất lượng phục vụ đối chiếu kết quả xử lý. Các bảng fact và dimension vẫn là dữ liệu đầu vào của EDA; báo cáo kiểm tra không được dùng để thay thế hoặc tính ngược lại các giá trị phân tích.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd() / "scripts"))
from common import (
    RAW, TABLES, QUALITY, ensure_output_dirs, table_quality, assert_unique_key,
    last_complete_month, sha256_file, write_csv_atomic, write_json_atomic,
)
ensure_output_dirs()

specs = {
    "dim_coin": ("dim_coin.csv", ["ticker"], "one ticker", "coin taxonomy, true price coverage and same-date current-cap rank"),
    "fact_crypto_daily": ("fact_crypto_daily.csv.gz", ["ticker", "date"], "one ticker-date", "valid-price crypto panel, on-chain measures and calculated fields"),
    "fact_market_daily": ("fact_market_daily.csv.gz", ["ticker", "date"], "one ticker-date", "separate current/estimated cap, reported spot volume and current-cap market share"),
    "fact_macro_daily": ("fact_macro_daily.csv", ["date"], "one date", "variables observed at true daily frequency"),
    "fact_macro_monthly": ("fact_macro_monthly.csv", ["month"], "one calendar month", "monthly source levels and correctly aggregated daily variables"),
    "analysis_crypto_macro_monthly": ("analysis_crypto_macro_monthly.csv", ["month"], "one complete calendar month", "analysis-ready BTC/ETH and macro transformations"),
    "dim_country": ("dim_country.csv", ["iso3"], "one ISO3", "country/economy/territory attributes"),
    "fact_country_yearly": ("fact_country_yearly.csv", ["iso3", "year"], "one ISO3-year", "annual inflation, GDP and population context"),
}
date_cols = {
    "dim_coin": ["raw_first_date", "raw_last_date", "price_first_date", "price_last_date", "market_cap_as_of_date"],
    "fact_crypto_daily": ["date"],
    "fact_market_daily": ["date"],
    "fact_macro_daily": ["date"],
    "fact_macro_monthly": ["month"],
    "analysis_crypto_macro_monthly": ["month"],
}
tables, registry_rows, quality_rows = {}, [], []
for table_name, (filename, key, grain, purpose) in specs.items():
    kwargs = {"parse_dates": date_cols.get(table_name), "low_memory": False}
    if table_name in {"dim_country", "fact_country_yearly"}:
        kwargs.update({"keep_default_na": False, "na_values": [""]})
    frame = pd.read_csv(TABLES / filename, **kwargs)
    tables[table_name] = frame
    quality_rows.append(table_quality(frame, table_name, key))
    registry_rows.append(
        {
            "table": table_name, "file": filename, "grain": grain,
            "key": ",".join(key), "purpose": purpose,
            "rows": len(frame), "columns": frame.shape[1],
        }
    )
registry = pd.DataFrame(registry_rows)
quality = pd.DataFrame(quality_rows)
display(registry)

,table,file,grain,key,purpose,rows,columns
0,dim_coin,dim_coin.csv,one ticker,ticker,"coin taxonomy, true price coverage and same-da...",63,16
1,fact_crypto_daily,fact_crypto_daily.csv.gz,one ticker-date,"ticker,date","valid-price crypto panel, on-chain measures an...",123358,29
2,fact_market_daily,fact_market_daily.csv.gz,one ticker-date,"ticker,date","separate current/estimated cap, reported spot ...",185694,11
3,fact_macro_daily,fact_macro_daily.csv,one date,date,variables observed at true daily frequency,10071,4
4,fact_macro_monthly,fact_macro_monthly.csv,one calendar month,month,monthly source levels and correctly aggregated...,2325,24
5,analysis_crypto_macro_monthly,analysis_crypto_macro_monthly.csv,one complete calendar month,month,analysis-ready BTC/ETH and macro transformations,160,31
6,dim_country,dim_country.csv,one ISO3,iso3,country/economy/territory attributes,249,10
7,fact_country_yearly,fact_country_yearly.csv,one ISO3-year,"iso3,year","annual inflation, GDP and population context",2580,23


## Kiểm tra miền giá trị, công thức và tính nhất quán

Các kiểm tra được tổ chức theo những điều kiện có thể đối chiếu trực tiếp:

- Giá crypto phải dương; vốn hóa không thiếu phải dương; volume trong bảng giá phải nằm trong khoảng 0–`10^15 USD/ngày`. Snapshot này phải có đúng ba dòng mang cờ volume không hợp lệ.
- Ticker trong hai bảng fact crypto phải có trong `dim_coin`; bảng quốc gia phải có mã châu lục.
- Bảng phân tích phải có lịch tháng liên tục và kết thúc đúng mốc tháng hoàn chỉnh xác định từ dữ liệu giá crypto.
- Với BTC, ETH và XRP nếu có dữ liệu, notebook tính lại lợi suất, MA7, drawdown và điểm robust z từ median/MAD quá khứ để so với trường đã lưu.
- CPI MoM/YoY được tính lại trên lịch tháng đầy đủ; tổng thị phần vốn hóa hiện tại được kiểm tra bằng 100%.
- Tỷ lệ volume/vốn hóa được đối chiếu với công thức. Vốn hóa hiện tại được so với giá nhân nguồn cung trên các dòng đủ cả ba trường, với sai số tương đối phải nhỏ hơn 0,5%.
- DXY lưu trong bảng ngày được so với kết quả dựng lại từ sáu tỷ giá nguồn; SHA-256 của các tệp dữ liệu thô được so với manifest.

Kiểm tra DXY xác nhận sự nhất quán với công thức và tỷ giá đang dùng, không phải đối chiếu với một chuỗi ICE độc lập. Tương tự, checksum xác nhận nội dung tệp còn khớp manifest, không đánh giá chất lượng kinh tế của dữ liệu.

In [2]:
crypto = tables["fact_crypto_daily"]
market = tables["fact_market_daily"]
macro = tables["fact_macro_monthly"].sort_values("month").set_index("month")
analysis = tables["analysis_crypto_macro_monthly"]
country = tables["fact_country_yearly"]
dim_coin = tables["dim_coin"]

assert crypto["price_usd"].gt(0).all()
assert crypto["market_cap_current_usd"].dropna().gt(0).all()
assert crypto["market_cap_estimated_usd"].dropna().gt(0).all()
assert crypto["reported_spot_volume_usd"].dropna().between(0, 1e15, inclusive="both").all()
assert crypto["reported_spot_volume_invalid_flag"].notna().all()
assert int(crypto["reported_spot_volume_invalid_flag"].sum()) == 3
assert market["market_cap_current_usd"].dropna().gt(0).all()
assert market["market_cap_estimated_usd"].dropna().gt(0).all()
assert country["continent_code"].notna().all()
assert set(crypto["ticker"]).issubset(set(dim_coin["ticker"]))
assert set(market["ticker"]).issubset(set(dim_coin["ticker"]))

expected_last_month = last_complete_month(crypto["date"].max())
assert analysis["month"].max() == expected_last_month
expected_analysis_calendar = pd.Series(
    pd.date_range(analysis["month"].min(), analysis["month"].max(), freq="ME"),
    name="month",
)
assert analysis["month"].reset_index(drop=True).equals(expected_analysis_calendar)

for ticker in [x for x in ["BTC", "ETH", "XRP"] if x in set(crypto["ticker"])]:
    part = crypto.loc[crypto["ticker"].eq(ticker)].sort_values("date")
    assert np.allclose(
        part["daily_return_pct"], part["price_usd"].pct_change(fill_method=None) * 100,
        equal_nan=True,
    )
    assert np.allclose(
        part["ma7_usd"], part["price_usd"].rolling(7, min_periods=7).mean(),
        equal_nan=True,
    )
    assert np.allclose(
        part["drawdown_pct"], (part["price_usd"] / part["price_usd"].cummax() - 1) * 100,
        equal_nan=True,
    )

    past = part["daily_return_pct"].shift(1)
    rolling = past.rolling(365, min_periods=60)
    expected_median = rolling.median()
    expected_mad = rolling.apply(
        lambda values: np.median(np.abs(values - np.median(values))), raw=True
    )
    expected_robust_z = (
        (part["daily_return_pct"] - expected_median)
        / (1.4826 * expected_mad.replace(0, np.nan))
    )
    assert np.allclose(
        part["return_robust_z_365"], expected_robust_z, equal_nan=True, rtol=1e-10, atol=1e-10
    )

expected_cpi_mom = (macro["cpi_index"] / macro["cpi_index"].shift(1) - 1) * 100
expected_cpi_yoy = (macro["cpi_index"] / macro["cpi_index"].shift(12) - 1) * 100
expected_macro_calendar = pd.DatetimeIndex(
    pd.date_range(macro.index.min(), macro.index.max(), freq="ME"), name="month"
)
assert macro.index.equals(expected_macro_calendar)
assert np.allclose(macro["cpi_mom_pct"], expected_cpi_mom, equal_nan=True)
assert np.allclose(macro["cpi_yoy_pct"], expected_cpi_yoy, equal_nan=True)

share_sum = (
    market.dropna(subset=["market_cap_current_usd"])
    .groupby("date")["current_market_share_available_pct"].sum()
)
assert np.allclose(share_sum, 100, atol=1e-8)

turnover_mask = crypto[["reported_spot_volume_usd", "market_cap_current_usd"]].notna().all(axis=1)
expected_turnover = (
    crypto.loc[turnover_mask, "reported_spot_volume_usd"]
    / crypto.loc[turnover_mask, "market_cap_current_usd"]
)
assert np.allclose(
    crypto.loc[turnover_mask, "turnover_ratio"], expected_turnover, rtol=1e-12, atol=1e-12
)

cap_mask = crypto[["market_cap_current_usd", "price_usd", "supply_current_native"]].notna().all(axis=1)
cap_relative_error = (
    crypto.loc[cap_mask, "market_cap_current_usd"]
    - crypto.loc[cap_mask, "price_usd"] * crypto.loc[cap_mask, "supply_current_native"]
).abs() / crypto.loc[cap_mask, "market_cap_current_usd"].abs()
assert cap_relative_error.max() < 0.005

fx = pd.read_csv(RAW / "macro" / "fx_daily.csv")
fx["Date"] = pd.to_datetime(fx["Date"], errors="coerce")
fx["Exchange rate"] = pd.to_numeric(fx["Exchange rate"], errors="coerce")
weights = {"Euro": .576, "Japan": .136, "United Kingdom": .119, "Canada": .091, "Sweden": .042, "Switzerland": .036}
fx_wide = fx.loc[fx["Country"].isin(weights) & fx["Date"].ge("2013-01-01")].pivot(
    index="Date", columns="Country", values="Exchange rate"
)[list(weights)].dropna()
expected_dxy = 50.14348112 * np.exp(
    sum(np.log(fx_wide[name]) * weight for name, weight in weights.items())
)
stored_dxy = tables["fact_macro_daily"].set_index("date")["dxy_reconstructed"]
dxy_check = pd.concat([expected_dxy.rename("expected"), stored_dxy.rename("stored")], axis=1).dropna()
assert np.allclose(dxy_check["expected"], dxy_check["stored"], rtol=1e-10, atol=1e-10)

manifest = pd.read_csv(RAW / "SOURCE_MANIFEST.csv")
manifest["computed_sha256"] = manifest["raw_path"].map(lambda x: sha256_file(RAW / x))
assert manifest["sha256"].eq(manifest["computed_sha256"]).all()
print("Primary keys, domains, exact rolling MAD, volume/cap formulas, CPI calendar, current-cap shares, DXY and raw checksums: PASS")

Primary keys, domains, exact rolling MAD, volume/cap formulas, CPI calendar, current-cap shares, DXY and raw checksums: PASS


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_5844\1855162781.py:99: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  dxy_check = pd.concat([expected_dxy.rename("expected"), stored_dxy.rename("stored")], axis=1).dropna()


## Từ điển dữ liệu và các báo cáo kiểm tra

Từ điển dữ liệu được tạo cho từng cặp bảng–cột, gồm định nghĩa, đơn vị, nguồn hoặc công thức, kiểu dữ liệu, số giá trị thiếu và số dòng. Nếu một cột trong tám bảng chưa có mục mô tả trong danh sách `D`, notebook dừng để yêu cầu bổ sung. Kiểm tra này bảo đảm có tài liệu cho từng cột; ý nghĩa của trường vẫn cần được đọc cùng công thức và quy ước xử lý ở các notebook trước.

Notebook đồng thời lưu các báo cáo trong `data/processed/quality_reports/`:

| Nhóm báo cáo | Nội dung |
|---|---|
| `processed_quality_summary.csv/.json` và `qa_summary.csv` | Tổng hợp chất lượng bảng và trạng thái các kiểm tra đã thực hiện |
| `table_registry.csv` và `data_dictionary_processed.csv` | Danh mục bảng, khóa, mục đích và định nghĩa từng cột |
| `cleaning_policy.csv` | Quy tắc xử lý giá trị thiếu, miền giá trị, ngoại lệ và phép nối |
| `source_processing_map.csv` | Đường đi từ từng tệp nguồn tới bảng xử lý |
| `join_lineage_detailed.csv` | Thông tin truy vết các phép nối, số dòng và khóa không khớp |
| `function_technique_catalog.csv` | Các hàm và kỹ thuật được sử dụng trong chuỗi notebook |

Trạng thái `PASS` chỉ được tạo sau khi các điều kiện kiểm tra phía trước đã vượt qua. Nó phản ánh phạm vi kiểm tra đã khai báo, không có nghĩa dữ liệu hoàn toàn không còn sai lệch hoặc mọi giả định thống kê đã được xác nhận.

In [3]:
D = {
    "date": ("Daily observation key.", "date", "raw/derived"),
    "month": ("Calendar month-end join key; not a claim that each source value is month-end.", "month", "derived"),
    "year": ("Calendar year key.", "year", "raw"),
    "ticker": ("Uppercase Coin Metrics asset symbol.", "symbol", "archive member name"),
    "price_usd": ("Positive Coin Metrics daily PriceUSD.", "USD/asset", "PriceUSD"),
    "market_cap_current_usd": ("Market cap using current supply; primary cap measure.", "USD", "CapMrktCurUSD"),
    "market_cap_estimated_usd": ("Estimated market cap retained separately; never silently substituted for current cap.", "USD", "CapMrktEstUSD"),
    "reported_spot_volume_usd": ("Reported one-day spot trading volume after nulling three documented >1e15 USD/day source-magnitude failures; not adjusted on-chain transfer value.", "USD/day", "volume_reported_spot_usd_1d"),
    "reported_spot_volume_invalid_flag": ("True for a raw reported-volume magnitude above 1e15 USD/day that was retained in raw but set missing in processed.", "boolean", "calculated validity rule"),
    "active_addresses": ("Active address count.", "addresses/day", "AdrActCnt"),
    "tx_count": ("Transaction count.", "transactions/day", "TxCnt"),
    "transfer_count": ("Transfer count.", "transfers/day", "TxTfrCnt"),
    "hashrate": ("Network hash rate where applicable.", "source units", "HashRate"),
    "supply_current_native": ("Current native-unit supply.", "native asset units", "SplyCur"),
    "mvrv_current": ("Current MVRV ratio.", "ratio", "CapMVRVCur"),
    "exchange_inflow_usd": ("Exchange inflow where source provides it.", "USD/day", "FlowInExUSD"),
    "exchange_outflow_usd": ("Exchange outflow where source provides it.", "USD/day", "FlowOutExUSD"),
    "log_return": ("Daily log difference of PriceUSD within ticker.", "log points", "calculated"),
    "daily_return_pct": ("Daily percentage change of PriceUSD within ticker.", "%", "calculated"),
    "ma7_usd": ("Seven-observation moving average of PriceUSD.", "USD/asset", "calculated"),
    "ma30_usd": ("Thirty-observation moving average of PriceUSD.", "USD/asset", "calculated"),
    "ma90_usd": ("Ninety-observation moving average of PriceUSD.", "USD/asset", "calculated"),
    "volatility_30d_annualized_pct": ("Rolling 30-observation std(log return)*sqrt(365)*100.", "% annualized", "calculated"),
    "drawdown_pct": ("Price/running maximum minus one, times 100.", "%", "calculated"),
    "turnover_ratio": ("Reported spot volume divided by current market cap.", "ratio/day", "calculated"),
    "return_robust_z_365": ("Past-only rolling median/MAD robust z-score of daily return.", "robust sd", "calculated"),
    "is_return_outlier": ("True when |return_robust_z_365| > 6; row retained.", "boolean", "calculated"),
    "is_up_day": ("True when non-missing daily return is positive.", "boolean", "calculated"),
    "total_current_market_cap_available_usd": ("Sum of available current caps for the observed universe and date.", "USD", "calculated"),
    "assets_with_current_market_cap": ("Assets contributing current cap on the date.", "count", "calculated"),
    "btc_current_market_cap_usd": ("BTC current market cap used in the dominance numerator.", "USD", "CapMrktCurUSD"),
    "btc_dominance_current_available_universe_pct": ("BTC current cap divided by current-cap total of available universe; not global dominance.", "%", "calculated"),
    "current_market_share_available_pct": ("Asset current cap divided by same-day current-cap total of available universe.", "%", "calculated"),
    "btc_price_usd": ("Last valid BTC PriceUSD in the calendar month.", "USD/BTC", "calculated"),
    "btc_return_pct": ("Change in successive BTC last-valid monthly prices.", "%", "calculated"),
    "eth_price_usd": ("Last valid ETH PriceUSD in the calendar month.", "USD/ETH", "calculated"),
    "eth_return_pct": ("Change in successive ETH last-valid monthly prices.", "%", "calculated"),
    "sp500_level": ("Source monthly S&P 500 price-level observation: Shiller history is a monthly average; the recent extension uses FRED SP500. Not a total-return index.", "index points", "SP500"),
    "sp500_pe10": ("Cyclically adjusted PE10 where source supplies a positive value.", "ratio", "PE10"),
    "sp500_return_pct": ("Change in successive source monthly S&P 500 price levels; not total return.", "%", "calculated"),
    "gold_usd_per_oz": ("Monthly World Bank gold price in the 2013+ analysis window; not relabeled month-end. Upstream pre-1960 values repeat annual averages by month.", "USD/troy oz", "Price"),
    "gold_return_pct": ("Change in successive source monthly gold price levels.", "%", "calculated"),
    "cpi_index": ("US CPI index level.", "index", "Index"),
    "cpi_mom_pct": ("CPI change from previous calendar month; missing if either month missing.", "%", "calculated"),
    "cpi_yoy_pct": ("CPI change from the same calendar month 12 months earlier.", "%", "calculated"),
    "cpi_yoy_change_pp": ("First difference of CPI YoY.", "percentage points", "calculated"),
    "us10y_yield_pct": ("Monthly US 10-year Treasury yield.", "%/year", "Rate"),
    "us10y_change_pp": ("First difference of US10Y yield.", "percentage points", "calculated"),
    "vix_close": ("Daily VIX close.", "index points", "CLOSE"),
    "vix_mean": ("Mean daily VIX close in month.", "index points", "calculated"),
    "vix_last": ("Last available daily VIX close in month.", "index points", "calculated"),
    "vix_max": ("Maximum daily VIX close in month.", "index points", "calculated"),
    "vix_observations": ("Non-missing daily VIX observations in month.", "trading days", "calculated"),
    "vix_change_pct": ("Change in successive monthly mean VIX.", "%", "calculated"),
    "risk_regime": ("Risk-on <15, Neutral 15 to <25, Risk-off >=25 using same-month mean VIX.", "category", "calculated convention"),
    "brent_usd_per_barrel": ("Daily or last-in-month Brent price, depending on table grain.", "USD/barrel", "Price"),
    "brent_return_pct": ("Change in successive last-in-month Brent prices.", "%", "calculated"),
    "dxy_reconstructed": ("DXY formula reconstructed from six foreign-currency-per-USD rates; not official ICE feed.", "index points", "calculated"),
    "dxy_return_pct": ("Change in successive last-in-month reconstructed DXY levels.", "%", "calculated"),
    "real_rate_pct": ("Ex-post proxy: nominal US10Y minus CPI YoY.", "%", "calculated"),
    "real_rate_change_pp": ("First difference of ex-post real-rate proxy.", "percentage points", "calculated"),
    "raw_first_date": ("First raw row date for ticker.", "date", "raw coverage"),
    "raw_last_date": ("Last raw row date for ticker.", "date", "raw coverage"),
    "raw_rows": ("Raw rows for ticker.", "rows", "raw coverage"),
    "price_first_date": ("First date with valid positive PriceUSD.", "date", "derived"),
    "price_last_date": ("Last date with valid positive PriceUSD.", "date", "derived"),
    "valid_price_rows": ("Rows with valid positive PriceUSD.", "rows", "derived"),
    "valid_market_rows": ("Rows with at least one cap or reported-volume field.", "rows", "derived"),
    "valid_current_cap_rows": ("Rows with valid current cap.", "rows", "derived"),
    "valid_estimated_cap_rows": ("Rows with valid estimated cap.", "rows", "derived"),
    "has_price_history": ("Whether ticker has any valid PriceUSD.", "boolean", "derived"),
    "category": ("Hand-curated descriptive asset category; not inferential.", "category", "manual taxonomy"),
    "category_source": ("Provenance label for category.", "text", "manual taxonomy"),
    "market_cap_current_as_of_usd": ("Current cap at the common as-of date.", "USD", "CapMrktCurUSD"),
    "market_cap_current_rank_as_of": ("Descending current-cap rank at the common as-of date.", "rank", "calculated"),
    "market_cap_as_of_date": ("Common date used for current-cap rank.", "date", "derived"),
    "iso3": ("ISO3 identifier after aggregate exclusion.", "code", "country_codes"),
    "iso2": ("ISO2 identifier.", "code", "country_codes"),
    "country_name": ("Country/economy/territory display name.", "text", "country_codes"),
    "region": ("UN region name.", "text", "country_codes"),
    "sub_region": ("UN sub-region name.", "text", "country_codes"),
    "continent_code": ("Continent code; NA means North America.", "code", "country_codes"),
    "capital": ("Capital name where available.", "text", "country_codes"),
    "currency_code": ("Alphabetic currency code.", "code", "country_codes"),
    "continent_name": ("Mapped continent name.", "text", "derived"),
    "is_ldc": ("Least Developed Country marker from dimension.", "0/1", "country_codes"),
    "inflation_annual_pct": ("Annual CPI inflation; cross-economy comparisons are descriptive.", "%/year", "World Bank FP.CPI.TOTL.ZG"),
    "gdp_current_usd": ("GDP in current USD.", "USD/year", "World Bank GDP"),
    "population": ("Annual population.", "persons", "World Bank population"),
    "gdp_per_capita_usd": ("Current-USD GDP divided by population for same ISO3-year.", "USD/person", "calculated"),
    "log_gdp_per_capita": ("Natural log of positive GDP per capita.", "log USD/person", "calculated"),
    "gdp_per_capita_growth_pct": ("Within-economy change in current-USD GDP per capita.", "%", "calculated"),
    "inflation_year_z": ("Cross-sectional inflation z-score within year.", "standard deviations", "calculated"),
    "inflation_extreme_flag": ("True when |inflation_year_z| > 4; row retained.", "boolean", "calculated"),
    "high_inflation_ge_10pct": ("Descriptive flag for annual inflation >=10%.", "boolean", "calculated convention"),
    "has_inflation": ("Inflation availability flag.", "0/1", "derived"),
    "has_gdp": ("GDP availability flag.", "0/1", "derived"),
    "has_population": ("Population availability flag.", "0/1", "derived"),
}

all_columns = sorted({column for frame in tables.values() for column in frame.columns})
undocumented = sorted(set(all_columns) - set(D))
assert not undocumented, f"Undocumented processed columns: {undocumented}"
dictionary_rows = []
for table_name, frame in tables.items():
    for column in frame.columns:
        definition, unit, source = D[column]
        dictionary_rows.append(
            {
                "table": table_name, "column": column, "definition": definition,
                "unit": unit, "source_or_formula": source,
                "dtype": str(frame[column].dtype), "missing": int(frame[column].isna().sum()),
                "rows": len(frame),
            }
        )
data_dictionary = pd.DataFrame(dictionary_rows)
display(data_dictionary.head(20))

,table,column,definition,unit,source_or_formula,dtype,missing,rows
0,dim_coin,ticker,Uppercase Coin Metrics asset symbol.,symbol,archive member name,str,0,63
1,dim_coin,raw_first_date,First raw row date for ticker.,date,raw coverage,datetime64[us],0,63
2,dim_coin,raw_last_date,Last raw row date for ticker.,date,raw coverage,datetime64[us],0,63
3,dim_coin,raw_rows,Raw rows for ticker.,rows,raw coverage,int64,0,63
4,dim_coin,price_first_date,First date with valid positive PriceUSD.,date,derived,datetime64[us],24,63
5,dim_coin,price_last_date,Last date with valid positive PriceUSD.,date,derived,datetime64[us],24,63
6,dim_coin,valid_price_rows,Rows with valid positive PriceUSD.,rows,derived,int64,0,63
7,dim_coin,valid_market_rows,Rows with at least one cap or reported-volume ...,rows,derived,int64,0,63
8,dim_coin,valid_current_cap_rows,Rows with valid current cap.,rows,derived,int64,0,63
9,dim_coin,valid_estimated_cap_rows,Rows with valid estimated cap.,rows,derived,int64,0,63


In [4]:
cleaning_policy = pd.DataFrame(
    [
        {"area": "crypto_price", "issue": "missing/non-positive PriceUSD", "detection": "isna or <=0", "treatment": "exclude only from price panel; no interpolation", "rationale": "Interpolation could create artificial returns."},
        {"area": "crypto_market_cap", "issue": "current and estimated cap differ conceptually", "detection": "inspect CapMrktCurUSD and CapMrktEstUSD separately", "treatment": "retain both; current only for rank/share/dominance", "rationale": "Avoid mixing current supply with estimated supply."},
        {"area": "crypto_volume", "issue": "ambiguous volume meaning", "detection": "raw field name review", "treatment": "rename to reported_spot_volume_usd; no imputation", "rationale": "It is reported spot volume, not adjusted on-chain transfer value."},
        {"area": "crypto_volume_magnitude", "issue": "three astronomically impossible USDC reported-volume magnitudes", "detection": ">1e15 USD/day plus neighboring-date review", "treatment": "flag; set processed value missing; preserve immutable raw", "rationale": "Values of 1e47-1e53 USD/day are source failures, not plausible market extremes."},
        {"area": "crypto_return_outlier", "issue": "extreme return", "detection": "past-only 365-observation median/MAD z; minimum 60; |z|>6", "treatment": "flag and retain", "rationale": "Extreme crypto moves can be real; robust estimator limits masking without look-ahead."},
        {"area": "macro_calendar", "issue": "missing calendar month", "detection": "source months versus complete MonthEnd index", "treatment": "insert calendar key and retain missing; no interpolation", "rationale": "Ensures shift(12) uses the same calendar month."},
        {"area": "macro_transformation", "issue": "trending levels", "detection": "time-series measurement design", "treatment": "use returns/changes for correlation", "rationale": "Reduces spurious correlation from common trends."},
        {"area": "vix_regime", "issue": "descriptive grouping", "detection": "monthly mean VIX", "treatment": "Risk-on <15; Neutral 15-<25; Risk-off >=25", "rationale": "Transparent ex-ante convention, not fitted to outcomes."},
        {"area": "country_key", "issue": "World Bank aggregates and NA code", "detection": "ISO3 join to country_codes", "treatment": "exclude unmapped aggregates; preserve NA as North America", "rationale": "Prevents double counting and false missing geography."},
        {"area": "country_inflation_outlier", "issue": "extreme annual inflation", "detection": "within-year |z|>4", "treatment": "flag and retain; symlog visualization", "rationale": "Hyperinflation is substantively meaningful."},
        {"area": "joins", "issue": "row multiplication/grain mismatch", "detection": "assert unique keys and validate cardinality", "treatment": "only compatible 1:1 or many:1 joins; country annual kept separate", "rationale": "Prevents invalid replication and totals."},
        {"area": "study_window", "issue": "window dependence", "detection": "pre-specified 2018 main window", "treatment": "also export 2015-start sensitivity", "rationale": "Focus on contemporary regime while exposing sensitivity."},
    ]
)

qa_summary = pd.DataFrame(
    [
        {"check": "raw_checksums", "status": "PASS", "detail": f"{len(manifest)} source files match manifest SHA-256"},
        {"check": "primary_keys", "status": "PASS", "detail": f"{len(specs)} tables have unique non-null keys"},
        {"check": "domain_constraints", "status": "PASS", "detail": "positive prices/caps, nonnegative reported volume, 3 impossible volume magnitudes flagged/null, valid geography"},
        {"check": "calculated_fields", "status": "PASS", "detail": "returns, MA7, drawdown, exact past-only rolling MAD, turnover, cap identity tolerance, CPI shifts, current-cap shares and DXY reproduced"},
        {"check": "cleaning_policy", "status": "PASS", "detail": f"{len(cleaning_policy)} explicit policies"},
        {"check": "data_dictionary", "status": "PASS", "detail": f"{len(data_dictionary)} table-column definitions; no undocumented columns"},
    ]
)

source_processing_map = pd.DataFrame(
    [
        {"raw_file": "crypto/coinmetrics_raw.tar.gz", "raw_grain_key": "ticker + time", "fields_used": "PriceUSD; CapMrktCurUSD; CapMrktEstUSD; volume_reported_spot_usd_1d; on-chain metrics", "cleaning_and_transform": "parse date/numeric; require positive price/cap/supply; nonnegative counts/flows; flag and null 3 volume values >1e15 USD/day; no imputation", "processed_destination": "dim_coin; fact_crypto_daily; fact_market_daily", "join_rule": "ticker-date remains unique; daily market context joins many-to-one on date", "caveat": "current and estimated cap stay separate; reported spot volume is not on-chain transfer value"},
        {"raw_file": "macro/vix_daily.csv", "raw_grain_key": "DATE", "fields_used": "CLOSE", "cleaning_and_transform": "numeric/date parse; positive close; monthly mean/last/max/count", "processed_destination": "fact_macro_daily; fact_macro_monthly; analysis_crypto_macro_monthly", "join_rule": "daily key kept; monthly aggregation joins one-to-one on month", "caveat": "VIX regime is a transparent project convention"},
        {"raw_file": "macro/brent_daily.csv", "raw_grain_key": "Date", "fields_used": "Price", "cleaning_and_transform": "numeric/date parse; positive price; last available trading observation per month; monthly percent change", "processed_destination": "fact_macro_daily; fact_macro_monthly; analysis_crypto_macro_monthly", "join_rule": "one-to-one on date/month after aggregation", "caveat": "monthly value is last available trading observation, not calendar-month average"},
        {"raw_file": "macro/fx_daily.csv", "raw_grain_key": "Date + Country", "fields_used": "six ICE-DXY component exchange rates", "cleaning_and_transform": "pivot; require all six rates; reconstruct geometric weighted DXY; last available monthly value", "processed_destination": "fact_macro_daily; fact_macro_monthly; analysis_crypto_macro_monthly", "join_rule": "one-to-one on date/month after reconstruction", "caveat": "proxy reconstructed from source rates; not official ICE feed"},
        {"raw_file": "macro/sp500_monthly.csv", "raw_grain_key": "Date/month", "fields_used": "SP500; PE10", "cleaning_and_transform": "complete calendar; nonpositive placeholders to missing; percent change without fill", "processed_destination": "fact_macro_monthly; analysis_crypto_macro_monthly", "join_rule": "one-to-one on month", "caveat": "Shiller history is monthly average; recent extension uses FRED SP500; price index, not total return"},
        {"raw_file": "macro/us10y_monthly.csv", "raw_grain_key": "Date/month", "fields_used": "Rate", "cleaning_and_transform": "complete calendar; first difference in percentage points; ex-post real-rate proxy with CPI YoY", "processed_destination": "fact_macro_monthly; analysis_crypto_macro_monthly", "join_rule": "one-to-one on month", "caveat": "real-rate field is an ex-post proxy, not expected real yield"},
        {"raw_file": "macro/gold_monthly.csv", "raw_grain_key": "Date/month", "fields_used": "Price", "cleaning_and_transform": "complete calendar; positive monthly level; percent change without fill", "processed_destination": "fact_macro_monthly; analysis_crypto_macro_monthly", "join_rule": "one-to-one on month", "caveat": "2013+ is World Bank monthly; upstream pre-1960 repeats annual averages"},
        {"raw_file": "macro/cpi_us_monthly.csv", "raw_grain_key": "Date/month", "fields_used": "Index", "cleaning_and_transform": "complete calendar; positive CPI; MoM shift(1), YoY shift(12), no interpolation", "processed_destination": "fact_macro_monthly; analysis_crypto_macro_monthly", "join_rule": "one-to-one on month", "caveat": "a missing source month stays missing and breaks affected change calculations"},
        {"raw_file": "country/wb_cpi.csv", "raw_grain_key": "Country Code + Year", "fields_used": "CPI", "cleaning_and_transform": "numeric year/value; retain legitimate inflation extremes; flag within-year |z|>4", "processed_destination": "fact_country_yearly", "join_rule": "outer one-to-one with GDP/population on ISO3-year", "caveat": "World Bank aggregates are removed through the country dimension"},
        {"raw_file": "country/wb_gdp.csv", "raw_grain_key": "Country Code + Year", "fields_used": "Value", "cleaning_and_transform": "numeric; nonpositive GDP to missing; combine with same-year population", "processed_destination": "fact_country_yearly", "join_rule": "outer one-to-one on ISO3-year", "caveat": "current USD, so growth mixes real growth, inflation and exchange rates"},
        {"raw_file": "country/wb_population.csv", "raw_grain_key": "Country Code + Year", "fields_used": "Value", "cleaning_and_transform": "numeric; nonpositive population to missing; calculate same-year GDP per capita", "processed_destination": "fact_country_yearly", "join_rule": "outer one-to-one on ISO3-year", "caveat": "no interpolation"},
        {"raw_file": "country/country_codes.csv", "raw_grain_key": "ISO3166-1-Alpha-3", "fields_used": "ISO codes; names; regions; continent; capital; currency; LDC", "cleaning_and_transform": "drop missing/duplicate ISO3; preserve NA as North America; map continent names", "processed_destination": "dim_country; fact_country_yearly", "join_rule": "many-to-one from country-year panel to ISO3 dimension", "caveat": "only ISO3 values present in the dimension are retained as countries/economies/territories"},
    ]
)

infl_raw = pd.read_csv(RAW / "country" / "wb_cpi.csv")[["Country Code", "Year"]].rename(columns={"Country Code": "iso3", "Year": "year"})
gdp_raw = pd.read_csv(RAW / "country" / "wb_gdp.csv")[["Country Code", "Year"]].rename(columns={"Country Code": "iso3", "Year": "year"})
pop_raw = pd.read_csv(RAW / "country" / "wb_population.csv")[["Country Code", "Year"]].rename(columns={"Country Code": "iso3", "Year": "year"})
ig = infl_raw.merge(gdp_raw, on=["iso3", "year"], how="outer", validate="one_to_one", indicator=True)
igp = ig.drop(columns="_merge").merge(pop_raw, on=["iso3", "year"], how="outer", validate="one_to_one", indicator=True)

daily_market_summary = market.sort_values("date").drop_duplicates("date")
igp_window = igp.loc[pd.to_numeric(igp["year"], errors="coerce").between(2013, 2024)].copy()
join_lineage = pd.DataFrame(
    [
        {"step": 1, "left_table": "coin metadata", "right_table": "same-date current-cap rank", "key": "ticker", "join_type": "left", "cardinality": "one_to_one", "left_rows_before": len(dim_coin), "right_rows_before": int(dim_coin["market_cap_current_as_of_usd"].notna().sum()), "rows_after": len(dim_coin), "left_unmatched": int(dim_coin["market_cap_current_as_of_usd"].isna().sum()), "right_unmatched": 0, "purpose": "attach a comparable rank from one common date", "scientific_decision": "never use each coin's own latest date or estimated cap"},
        {"step": 2, "left_table": "daily current-cap summary", "right_table": "BTC current cap", "key": "date", "join_type": "left", "cardinality": "one_to_one", "left_rows_before": len(daily_market_summary), "right_rows_before": int(market.loc[market["ticker"].eq("BTC") & market["market_cap_current_usd"].notna(), "date"].nunique()), "rows_after": len(daily_market_summary), "left_unmatched": int(daily_market_summary["btc_current_market_cap_usd"].isna().sum()), "right_unmatched": 0, "purpose": "form BTC dominance from a same-date numerator and denominator", "scientific_decision": "never carry BTC cap across dates"},
        {"step": 3, "left_table": "market_clean", "right_table": "daily current-cap summary", "key": "date", "join_type": "left", "cardinality": "many_to_one", "left_rows_before": len(market), "right_rows_before": market["date"].nunique(), "rows_after": len(market), "left_unmatched": int(market["total_current_market_cap_available_usd"].isna().sum()), "right_unmatched": 0, "purpose": "calculate same-universe market share and BTC dominance", "scientific_decision": "denominator is explicitly the available current-cap universe"},
        {"step": 4, "left_table": "crypto_base", "right_table": "daily current-cap summary", "key": "date", "join_type": "left", "cardinality": "many_to_one", "left_rows_before": len(crypto), "right_rows_before": market["date"].nunique(), "rows_after": len(crypto), "left_unmatched": int(crypto["total_current_market_cap_available_usd"].isna().sum()), "right_unmatched": 0, "purpose": "attach market context to every valid-price asset-day", "scientific_decision": "date-level context cannot multiply ticker-date rows"},
        {"step": 5, "left_table": "complete month calendar", "right_table": "BTC last-valid monthly price", "key": "month", "join_type": "left", "cardinality": "one_to_one", "left_rows_before": len(analysis), "right_rows_before": int(analysis["btc_price_usd"].notna().sum()), "rows_after": len(analysis), "left_unmatched": int(analysis["btc_price_usd"].isna().sum()), "right_unmatched": 0, "purpose": "anchor the analysis to an explicit calendar", "scientific_decision": "exclude the current incomplete month"},
        {"step": 6, "left_table": "analysis month calendar", "right_table": "ETH last-valid monthly price", "key": "month", "join_type": "left", "cardinality": "one_to_one", "left_rows_before": len(analysis), "right_rows_before": int(analysis["eth_price_usd"].notna().sum()), "rows_after": len(analysis), "left_unmatched": int(analysis["eth_price_usd"].isna().sum()), "right_unmatched": 0, "purpose": "add the ETH comparison at the same monthly grain", "scientific_decision": "last valid observation is selected inside each calendar month"},
        {"step": 7, "left_table": "analysis crypto monthly", "right_table": "market monthly snapshot", "key": "month", "join_type": "left", "cardinality": "one_to_one", "left_rows_before": len(analysis), "right_rows_before": int(analysis["total_current_market_cap_available_usd"].notna().sum()), "rows_after": len(analysis), "left_unmatched": int(analysis["total_current_market_cap_available_usd"].isna().sum()), "right_unmatched": 0, "purpose": "attach last-valid same-month market context", "scientific_decision": "monthly market context remains one row per month"},
        {"step": 8, "left_table": "analysis crypto monthly", "right_table": "fact_macro_monthly", "key": "month", "join_type": "left", "cardinality": "one_to_one", "left_rows_before": len(analysis), "right_rows_before": len(macro), "rows_after": len(analysis), "left_unmatched": int((~analysis["month"].isin(macro.index)).sum()), "right_unmatched": int((~macro.index.isin(analysis["month"])).sum()), "purpose": "align transformed crypto and macro measures at monthly grain", "scientific_decision": "country-year data is deliberately not joined here"},
        {"step": 9, "left_table": "wb_cpi", "right_table": "wb_gdp", "key": "iso3 + year", "join_type": "outer", "cardinality": "one_to_one", "left_rows_before": len(infl_raw), "right_rows_before": len(gdp_raw), "rows_after": len(ig), "left_unmatched": int(ig["_merge"].eq("left_only").sum()), "right_unmatched": int(ig["_merge"].eq("right_only").sum()), "purpose": "retain country-years available in either indicator", "scientific_decision": "missing values remain missing; no fabricated inner-join balance"},
        {"step": 10, "left_table": "inflation_gdp outer panel", "right_table": "wb_population", "key": "iso3 + year", "join_type": "outer", "cardinality": "one_to_one", "left_rows_before": len(ig), "right_rows_before": len(pop_raw), "rows_after": len(igp), "left_unmatched": int(igp["_merge"].eq("left_only").sum()), "right_unmatched": int(igp["_merge"].eq("right_only").sum()), "purpose": "enable same-country same-year GDP per capita", "scientific_decision": "GDP/population are divided only after exact key alignment"},
        {"step": 11, "left_table": "country indicator panel 2013-2024", "right_table": "dim_country", "key": "iso3", "join_type": "filter then left", "cardinality": "many_to_one", "left_rows_before": len(igp_window), "right_rows_before": len(tables["dim_country"]), "rows_after": len(country), "left_unmatched": int((~igp_window["iso3"].isin(tables["dim_country"]["iso3"])).sum()), "right_unmatched": int((~tables["dim_country"]["iso3"].isin(country["iso3"])).sum()), "purpose": "attach geography and remove World Bank aggregates", "scientific_decision": "country panel stays annual and separate from monthly market analysis"},
    ]
)

function_technique_catalog = pd.DataFrame(
    [
        {"notebook": "00", "function_or_technique": "tarfile + schema profiling + SHA-256", "use": "inventory archive members, columns, keys and immutable-source hashes", "reason": "provenance and reproducibility before cleaning"},
        {"notebook": "01", "function_or_technique": "pd.to_datetime / pd.to_numeric(errors='coerce')", "use": "standardize time and numeric types", "reason": "invalid parses become explicit missing values"},
        {"notebook": "01", "function_or_technique": "Series.where", "use": "enforce positive/nonnegative domains and null impossible volume magnitude", "reason": "preserve row/key while preventing invalid values from entering formulas"},
        {"notebook": "01", "function_or_technique": "merge(validate='one_to_one')", "use": "attach one common-date cap rank to dim_coin", "reason": "detect duplicate keys and prevent row multiplication"},
        {"notebook": "02", "function_or_technique": "complete MonthEnd reindex", "use": "insert explicit missing months", "reason": "shift(12) means same calendar month, not twelfth prior row"},
        {"notebook": "02", "function_or_technique": "resample('ME').agg / last", "use": "aggregate true daily VIX and select last trading observations for Brent/DXY", "reason": "aggregation matches each variable's analytical meaning"},
        {"notebook": "02", "function_or_technique": "geometric weighted DXY reconstruction", "use": "combine six foreign-currency-per-USD rates", "reason": "match index direction and fixed published weights while labeling it a proxy"},
        {"notebook": "03", "function_or_technique": "outer merge on ISO3-year", "use": "combine inflation, GDP and population without discarding partial coverage", "reason": "missing indicators stay visible and same-year alignment is enforced"},
        {"notebook": "03", "function_or_technique": "groupby(year).transform z-score", "use": "flag cross-sectional inflation extremes", "reason": "retain substantive hyperinflation while making extremes auditable"},
        {"notebook": "04", "function_or_technique": "groupby(ticker) + pct_change/diff", "use": "calculate returns within asset only", "reason": "prevent boundaries between tickers from contaminating returns"},
        {"notebook": "04", "function_or_technique": "rolling mean/std/cummax", "use": "moving averages, annualized volatility and drawdown", "reason": "time-ordered market descriptors"},
        {"notebook": "04", "function_or_technique": "shift(1) + exact rolling median/MAD", "use": "robust return-outlier score", "reason": "past-only estimator avoids look-ahead and resists extreme-return masking"},
        {"notebook": "04", "function_or_technique": "merge cardinality validation", "use": "attach daily and monthly context", "reason": "make grain compatibility executable, not just documentary"},
        {"notebook": "05", "function_or_technique": "assert / np.allclose", "use": "independently reproduce keys, formulas, shares, DXY and calendar shifts", "reason": "fail fast when a processed output is inconsistent"},
        {"notebook": "06-08", "function_or_technique": "Matplotlib / Seaborn static charts", "use": "coverage, time series, distributions, drawdown, ranking, heatmap, scatter and regimes", "reason": "chart form is matched to the analytical question and variable scale"},
        {"notebook": "08", "function_or_technique": "Pearson + Spearman + Benjamini-Hochberg", "use": "measure linear/monotonic association and control multiple-testing FDR", "reason": "avoid treating many unadjusted p-values as independent discoveries"},
        {"notebook": "08", "function_or_technique": "statsmodels OLS with HAC(3)", "use": "descriptive BTC-S&P slope with robust standard error", "reason": "reduce sensitivity to heteroskedasticity and short autocorrelation; not causal"},
    ]
)

write_csv_atomic(source_processing_map, QUALITY / "source_processing_map.csv")
write_csv_atomic(join_lineage, QUALITY / "join_lineage_detailed.csv")
write_csv_atomic(function_technique_catalog, QUALITY / "function_technique_catalog.csv")

write_csv_atomic(quality, QUALITY / "processed_quality_summary.csv")
write_json_atomic(quality.to_dict(orient="records"), QUALITY / "processed_quality_summary.json")
write_csv_atomic(registry, QUALITY / "table_registry.csv")
write_csv_atomic(data_dictionary, QUALITY / "data_dictionary_processed.csv")
write_csv_atomic(cleaning_policy, QUALITY / "cleaning_policy.csv")
write_csv_atomic(qa_summary, QUALITY / "qa_summary.csv")
display(cleaning_policy)
display(qa_summary)

,area,issue,detection,treatment,rationale
0,crypto_price,missing/non-positive PriceUSD,isna or <=0,exclude only from price panel; no interpolation,Interpolation could create artificial returns.
1,crypto_market_cap,current and estimated cap differ conceptually,inspect CapMrktCurUSD and CapMrktEstUSD separa...,retain both; current only for rank/share/domin...,Avoid mixing current supply with estimated sup...
2,crypto_volume,ambiguous volume meaning,raw field name review,rename to reported_spot_volume_usd; no imputation,"It is reported spot volume, not adjusted on-ch..."
3,crypto_volume_magnitude,three astronomically impossible USDC reported-...,>1e15 USD/day plus neighboring-date review,flag; set processed value missing; preserve im...,Values of 1e47-1e53 USD/day are source failure...
4,crypto_return_outlier,extreme return,past-only 365-observation median/MAD z; minimu...,flag and retain,Extreme crypto moves can be real; robust estim...
5,macro_calendar,missing calendar month,source months versus complete MonthEnd index,insert calendar key and retain missing; no int...,Ensures shift(12) uses the same calendar month.
6,macro_transformation,trending levels,time-series measurement design,use returns/changes for correlation,Reduces spurious correlation from common trends.
7,vix_regime,descriptive grouping,monthly mean VIX,Risk-on <15; Neutral 15-<25; Risk-off >=25,"Transparent ex-ante convention, not fitted to ..."
8,country_key,World Bank aggregates and NA code,ISO3 join to country_codes,exclude unmapped aggregates; preserve NA as No...,Prevents double counting and false missing geo...
9,country_inflation_outlier,extreme annual inflation,within-year |z|>4,flag and retain; symlog visualization,Hyperinflation is substantively meaningful.


,check,status,detail
0,raw_checksums,PASS,12 source files match manifest SHA-256
1,primary_keys,PASS,8 tables have unique non-null keys
2,domain_constraints,PASS,"positive prices/caps, nonnegative reported vol..."
3,calculated_fields,PASS,"returns, MA7, drawdown, exact past-only rollin..."
4,cleaning_policy,PASS,12 explicit policies
5,data_dictionary,PASS,148 table-column definitions; no undocumented ...
